# perceptome v0.2.0 — tutorial

End-to-end walkthrough of the four canonical workflows on a real public dataset (10x Genomics PBMC3K, downloaded automatically by scanpy). Runs in <2 minutes on a laptop.

The four workflows answer four kinds of question:

1. **Geometry** — *where do my cells live in module space?*
2. **Perceptivity** — *what can my cells do?* (capacity, headroom, saturation)
3. **Reference comparison** — *how does my dataset relate to known directions?* (cancer attractor, disease, aging)
4. **Drug screen** — *do these drugs engage one of the 9 validated mechanism-pathway anchors?*

This notebook is generated by `scripts/08_build_tutorial_notebook.py` and re-executed on every release.

## Setup

In [1]:
import scanpy as sc
import perceptome as pct
import pandas as pd
import numpy as np

print(f"perceptome version: {pct.__version__}")
print(f"scanpy version: {sc.__version__}")
print(f"modules in catalog: {len(pct.list_modules())}")

perceptome version: 0.2.0
scanpy version: 1.12.1
modules in catalog: 44


/var/folders/_7/gjr6kqyj13gfjrv71fk_9ky00000gn/T/ipykernel_76975/2969628563.py:7: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  print(f"scanpy version: {sc.__version__}")


## Load PBMC3K — 2,638 PBMCs from a healthy donor

We use the **raw** PBMC3K (full gene set, ~13K genes) for module scoring, and pull cell type labels from the processed version (1,838 HVG-filtered) by matching barcodes. This mirrors a typical real-world workflow where you have raw counts plus an annotation.

In [2]:
# Raw counts
adata = sc.datasets.pbmc3k()
print(f"raw: {adata.n_obs} cells × {adata.n_vars} genes")

# Annotated reference for cell type labels
adata_proc = sc.datasets.pbmc3k_processed()
print(f"processed: {adata_proc.n_obs} cells × {adata_proc.n_vars} genes (HVG-filtered)")

# Transfer cell_type labels by barcode
adata = adata[adata.obs_names.isin(adata_proc.obs_names)].copy()
adata.obs["cell_type"] = adata_proc.obs["louvain"].reindex(adata.obs_names).astype(str)
print(f"after barcode match: {adata.n_obs} cells, {adata.obs.cell_type.nunique()} cell types")

# Standard log-normalize (perceptome expects log-normalized expression in .X)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
print(f"normalized: max={adata.X.max():.2f} (log scale)")

raw: 2700 cells × 32738 genes


processed: 2638 cells × 1838 genes (HVG-filtered)
after barcode match: 2638 cells, 8 cell types


normalized: max=7.47 (log scale)


## Workflow 1 — Geometry

Score 44 modules per cell, then project into the 9-PC HPA-derived eigenspace. PC1 is preserved from Paper 3 (perception breadth, myeloid+ to germ−); PC4 carries the cancer convergence direction (Paper 4.2).

In [3]:
scoring = pct.score_modules(adata, method="mean_raw")
scores = scoring["scores"]
print(f"scores: {scores.shape} (cells × modules)")

# Show gene coverage — how many catalog genes are present in this dataset?
cov = scoring["coverage"]
print(f"gene coverage: median {cov['n_genes_found'].median():.0f}/{cov['n_genes_total'].median():.0f} per module")
print(f"modules with no coverage: {(cov['n_genes_found'] == 0).sum()}")

# Top 5 modules in B cells (sanity check — should see immune-relevant signals)
b_mean = scores[adata.obs.cell_type == "B cells"].mean().nlargest(5)
print(f"\nTop 5 modules in B cells (by mean readiness):")
print(b_mean.round(2).to_string())

scores: (2638, 44) (cells × modules)
gene coverage: median 5/5 per module
modules with no coverage: 0

Top 5 modules in B cells (by mean readiness):
ERK/MAPK    1.05
GR          0.96
HSF1        0.30
Calcium     0.27
UPR-PERK    0.26


In [4]:
# Project per-cell scores into the canonical 9-PC eigenspace
proj = pct.project(scores)
coords = proj["coordinates"]
print(f"coordinates: {coords.shape}")
print(f"PC confidence: {proj['confidence']}")

# Per-cell-type centroids in PC1 vs PC4
ct_centroids = coords.groupby(adata.obs.cell_type).mean()
print("\nPer-cell-type centroids (PC1, PC4):")
print(ct_centroids[["PC1", "PC4"]].round(2).to_string())

coordinates: (2638, 9)
PC confidence: {'PC1': 'stable', 'PC2': 'stable', 'PC3': 'stable', 'PC4': 'stable', 'PC5': 'stable', 'PC6': 'stable', 'PC7': 'probable', 'PC8': 'probable', 'PC9': 'probable'}

Per-cell-type centroids (PC1, PC4):
                    PC1   PC4
cell_type                    
B cells           -0.91 -0.04
CD14+ Monocytes   -1.23 -0.01
CD4 T cells       -1.09 -0.02
CD8 T cells       -1.06 -0.02
Dendritic cells   -1.34 -0.03
FCGR3A+ Monocytes -1.25 -0.05
Megakaryocytes    -0.63 -0.05
NK cells          -1.15 -0.02


**Reading the centroids:** PC1 ranks cell types by perception breadth. Negative PC1 = germ-like (low module engagement), positive PC1 = myeloid-like (high engagement). PC4 carries the cancer convergence axis (Paper 4.2). Different blood cell types should arrange along PC1 in a biologically sensible order.

## Workflow 2 — Perceptivity (capacity layer)

The capacity layer answers a different question. Geometry tells you *where* a cell sits. Perceptivity tells you *what it can do from there* — which modules have ramp room (capacity) and which are saturated.

Two factors:
- **R (readiness)** — mean log expression of `core_genes` per module (machinery present)
- **A (activity)** — mean log expression of `activity_genes` per module (TF targets engaged)

Then `C = R − A` (capacity), `headroom = A_max − A` (distance to observed max), and `capacity_floor` (saturated/capacious/intermediate).

In [5]:
R = pct.score_readiness(adata)
A = pct.score_activity(adata)
print(f"R: {R.shape}, A: {A.shape}")

# IMPORTANT — threshold calibration: the default A_threshold=2.0 is calibrated for
# HPA pseudobulk scale (mean log1p(nCPM) of activity genes per cell type ≈ 0-7).
# Single-cell mean_raw scoring is on a much lower scale (≈ 0-2). For per-cell-derived
# perceptivity, use a lower threshold or compare against the bundled HPA reference
# directly (which IS at the right scale — see predict_engagement below).
perc = pct.compute_perceptivity(
    R, A,
    cell_type=adata.obs.cell_type,
    cell_class=adata.obs.cell_type,  # only one class here; BS will be NaN
    A_threshold=0.3,  # lowered for single-cell scale
)
per_ct = perc["per_cell_type"]
print(f"\nPer-cell-type perceptivity (A_threshold=0.3 for single-cell scale):")
print(per_ct[["n_engaged", "I_mean", "GS", "spec_quadrant"]].round(2).to_string())

# Compare module engagement counts across cell types — myeloid lineage should engage
# more modules than lymphoid, matching the perception breadth axis.
print(f"\nA-baseline matrix (per cell type × top-engaged modules):")
A_ct = perc["per_module"]["A"]
top_engaged = perc["per_module"]["engaged_mask"].sum(axis=0).nlargest(8).index.tolist()
print(A_ct[top_engaged].round(2).to_string())

R: (2638, 44), A: (2638, 44)

Per-cell-type perceptivity (A_threshold=0.3 for single-cell scale):
                   n_engaged  I_mean    GS                  spec_quadrant
cell_type                                                                
B cells                    7    0.56  0.86                 Q4: generalist
CD14+ Monocytes            9    0.70  2.43  Q1: locally + globally unique
CD4 T cells                8    0.57  1.43                 Q4: generalist
CD8 T cells                8    0.55  1.43                 Q4: generalist
Dendritic cells           11    0.63  3.57  Q1: locally + globally unique
FCGR3A+ Monocytes         11    0.64  3.29  Q1: locally + globally unique
Megakaryocytes             4    0.52  1.00                 Q4: generalist
NK cells                  11    0.48  3.29  Q1: locally + globally unique

A-baseline matrix (per cell type × top-engaged modules):
                   ERK/MAPK    GR  NRF2  Calcium  cAMP/CREB  mTOR  Type I IFN  UPR-ATF6
cell_type       

### A-priori prediction from HPA reference (no user data needed)

`predict_engagement` queries the bundled 154 × 44 HPA perceptivity reference, which IS calibrated for the absolute thresholds (A_baseline > 4.5 = saturated, < 2.5 = capacious). This is the most reliable way to use the capacity-floor predictor: ask "what's the capacity profile of cell type X for module Y" before running any experiment.

Example: a paper4.5 a-priori prediction was "enteric stem cells UPR-ATF6 should be saturated" → this was confirmed empirically (Goblet UPR-ATF6 d=−0.10):

In [6]:
# Paper 4.5 a-priori case
pred = pct.predict_engagement("enteric stem cells", ["UPR-ATF6", "HSF1", "mTOR"])
print("Paper 4.5 ISC prediction:")
print(pred[["A_baseline", "headroom", "capacity_floor", "predicted_direction"]].to_string())

Paper 4.5 ISC prediction:
          A_baseline  headroom        capacity_floor       predicted_direction
module                                                                        
UPR-ATF6    5.579517  1.700189  saturated_blocked_up  up_blocked_down_possible
HSF1        3.219218  1.681212          intermediate    intermediate_uncertain
mTOR        4.518348  0.998615  saturated_blocked_up  up_blocked_down_possible


In [7]:
# Paper 4.4 a-priori case — cardiomyocyte HSF1 should be capacious (paper4.4 PASS d=+1.29)
pred = pct.predict_engagement("cardiomyocytes", ["HSF1", "UPR-ATF6", "mTOR"])
print("Paper 4.4 cardiomyocyte prediction:")
print(pred[["A_baseline", "headroom", "capacity_floor", "predicted_direction"]].to_string())

Paper 4.4 cardiomyocyte prediction:
          A_baseline  headroom capacity_floor     predicted_direction
module                                                               
HSF1        2.032003  2.868427      capacious             up_possible
UPR-ATF6    3.119641  4.160066   intermediate  intermediate_uncertain
mTOR        3.174475  2.342488   intermediate  intermediate_uncertain


**Interpreting the floor classifier (upward-asymmetric, closed paper4.5+4.6+4.7+4.8):**

- `A_baseline > 4.5` → **saturated_blocked_up**: module cannot ramp UP further. Active downward suppression by specific signaling (e.g. retinoic acid → UPR) IS still possible.
- `A_baseline < 2.5` → **capacious**: ramp possible; magnitude depends on operation intensity (factor 2, not in the tool yet).
- `2.5 ≤ A ≤ 4.5` → **intermediate**.

Two-factor framework: `engagement = capacity (factor 1, this tool) × operation_intensity (factor 2, your context modeling)`.

## Workflow 3 — Reference comparison

How does your sample relate to known directions in module space — cancer attractor (Paper 4.2 P3), disease vectors (RA / AD / IPF / DKD), aging axes (inflammaging / collapse)?

In [8]:
# Use B-cell centroid as a query point
b_centroid = coords[adata.obs.cell_type == "B cells"].mean().to_frame().T
b_centroid.index = ["B_cells_centroid"]

refs = pct.compare_to_references(b_centroid)
print("Reference comparison for B cells:")
print(f"  cos_attractor (Paper 4.2 cancer direction):     {refs['cos_attractor']['B_cells_centroid']:+.3f}")
print(f"  cos_inflammaging (blood old−young):              {refs['cos_inflammaging']['B_cells_centroid']:+.3f}")
print(f"  cos_collapse (bone marrow old−young):            {refs['cos_collapse']['B_cells_centroid']:+.3f}")
print(f"\n  Disease cosines:")
for d in refs["disease_similarities"].columns:
    print(f"    {d}: {refs['disease_similarities'][d]['B_cells_centroid']:+.3f}")

Reference comparison for B cells:
  cos_attractor (Paper 4.2 cancer direction):     -0.645
  cos_inflammaging (blood old−young):              +0.994
  cos_collapse (bone marrow old−young):            +0.120

  Disease cosines:
    RA: -0.809
    AD: +0.485
    IPF: +0.669
    DKD: -0.262


### Cancer attractor alignment

If your data is a tumor-vs-normal paired comparison, you can test alignment with the cancer-transformation direction directly using `attractor_alignment`. Sun 2021 paired HCC PASSed at 4/6 cell types > +0.20.

In [9]:
# Demo: build a synthetic shift vector and test alignment.
# In real use: shift = (tumor_scores.mean() - normal_scores.mean()) at the module level,
# from your own paired cohort. Here we just demonstrate the API.
attr = pct.load_attractor_direction()
synthetic_shift = pd.Series(
    {m: 0.5 * attr["attractor_direction_modules"][m] + np.random.normal(scale=0.1)
     for m in pct.list_modules()},
    name="shift"
)
np.random.seed(42)
result = pct.attractor_alignment(synthetic_shift, mode="modules")
print(f"attractor alignment (synthetic):")
print(f"  cosine: {result['cosine']:+.3f}")
print(f"  passes P3 threshold (>0.20): {result['passes_p3_threshold']}")

attractor alignment (synthetic):
  cosine: +0.900
  passes P3 threshold (>0.20): True


## Workflow 4 — Drug screen (narrow validated scope)

**Read this before using.** Paper 4.1 closed 2026-05-09 with **6 surviving validated drug findings + 11 pre-registered falsifications**. The tool supports ONLY the validated operation: activity-layer scoring of TF-target panels for 9 specific (class, module) anchors.

Falsified operations the tool deliberately does NOT support: drug-disease cosine matching, mechanism deconvolution from panel geometry, equilibrium framings, snapshot-pulsatile classifications, etc. See `docs/SCOPE.md` and `PAPER4_1_CANONICAL_RESULTS.md` for the full audit trail.

In [10]:
# View the 9 validated anchors with their full validation chain
anchors = pct.drug_anchors()
print("9 validated anchors from Paper 4.1:")
print(anchors[["class", "module", "expected_sign", "role", "block5_z", "block5_q", "block4_LD"]].to_string(index=False))

9 validated anchors from Paper 4.1:
      class     module  expected_sign             role  block5_z  block5_q  block4_LD
       MEKi   ERK/MAPK             -1 validated_rescue    -1.026    0.0002       4.26
Proteasomei   UPR-PERK              1 validated_rescue     1.209    0.0002        NaN
       CDKi Cell Cycle             -1 validated_rescue       NaN       NaN       3.30
      EGFRi   ERK/MAPK             -1 validated_rescue       NaN       NaN       1.07
      PI3Ki  PI3K/PTEN              1 validated_rescue       NaN       NaN       0.81
       IKKi      NF-κB             -1 validated_rescue       NaN       NaN       0.79
     HSP90i       HSF1              1 positive_control     2.149    0.0002        NaN
     HSP90i   UPR-ATF6              1 positive_control     0.736    0.0004        NaN
     HSP90i       NRF2              1 positive_control     0.510    0.0380        NaN


In [11]:
# Just the 6 validated rescues (drop the 3 stress-axis controls)
rescues = pct.drug_anchors(role="validated_rescue")
print("6 validated mechanism-pathway rescues:")
print(rescues[["class", "module", "expected_sign"]].to_string(index=False))

6 validated mechanism-pathway rescues:
      class     module  expected_sign
       MEKi   ERK/MAPK             -1
Proteasomei   UPR-PERK              1
       CDKi Cell Cycle             -1
      EGFRi   ERK/MAPK             -1
      PI3Ki  PI3K/PTEN              1
       IKKi      NF-κB             -1


To run the actual screen on your data, you need a CMap-style dataset (perturbation signatures × genes, with `pert_iname` annotation). PBMC3K is not a perturbation experiment, so we don't run the screen here. The signature for `pct.activity_layer_screen()` looks like:

```python
result = pct.activity_layer_screen(
    adata,
    pert_col="pert_iname",
    test_perturbations={"my_MEKi": ["trametinib", "selumetinib"]},
    panels="all_validated",  # all 9 anchors, or a custom (class, module, sign) list
    n_perm=10000,
    seed=42,
)
# Returns DataFrame with observed_z, p_one_sided, q_BH (BH-FDR), verdict
```

Background null = all `pert_iname` values not in `test_perturbations`. Need ≥30 background drugs (~1000 in the validated Paper 4.1 procedure).

## Validity scorecard — run before trusting any perturbation effect

If you have a perturbation-vs-control comparison, run the validity scorecard *before* interpreting module-level deltas. The scorecard catches the most common analysis-breaking artifacts. This was the lesson of the Paper 4.5 v1.2 amendment, where a baseline-shift artifact was caught by a random-200 null and the methodology switched to `scanpy_corrected`.

In [12]:
# Demo: artificially split PBMCs into "drug" and "ctrl" by cell index — this is NOT
# a real perturbation, just to show the API. The scorecard should report something other
# than a clean PASS because the split is not biologically meaningful.
adata.obs["fake_treatment"] = ["drug" if i % 2 == 0 else "ctrl" for i in range(adata.n_obs)]

sc_obj = pct.validate_perturbation(
    adata,
    condition_col="fake_treatment",
    perturbation_value="drug",
    control_value="ctrl",
    require_cell_cycle=False,  # PBMC3K isn't a perturbation that should hit cell cycle
)
print(pct.scorecard(sc_obj))

Overall: MIXED
Checks:
  ✓ random_200     value=-0.0006  (< 0.10)  → PASS
      ↳ baseline-shift artifact detector; FAIL ⇒ retry with score_method='scanpy_corrected' (paper4.5 v1.2 fix)
  ✓ housekeeping   value=-0.0076  (< 0.20)  → PASS
      ↳ technical normalization sanity
  ✗ cell_cycle     value=-0.0014  (> 0.30)  → FAIL
      ↳ positive control; FAIL ⇒ perturbation may not reach biology


**Reading the scorecard:** in real use, you want all three checks PASS *before* trusting per-module deltas. If `random_200` FAILs → ARTIFACT, retry with `score_method='scanpy_corrected'` (the paper4.5 v1.2 fix). If `cell_cycle` FAILs → INCONCLUSIVE, the perturbation may not have engaged biology at all.

## Summary

You've now seen all four workflows in action:

| Workflow | When to use it |
|---|---|
| **Geometry** (`score_modules` + `project`) | Position cells in 9-PC perceptual space; compare to other cells / cell types |
| **Perceptivity** (`compute_perceptivity` + `predict_engagement`) | Predict capacity / saturation a priori for a known operation; understand cell-type capacity profile |
| **Reference comparison** (`compare_to_references` + `attractor_alignment`) | Test alignment with cancer attractor, disease vectors, aging axes |
| **Drug screen** (`drug_anchors` + `activity_layer_screen`) | Test if drugs engage one of 9 validated mechanism-pathway anchors |

Plus the **validity scorecard** (`validate_perturbation`) — always run before interpreting perturbation effects.

## Where to go next

- [`docs/QUICKSTART.md`](../docs/QUICKSTART.md) — written form of this tutorial with extra notes on common pitfalls
- [`docs/SCOPE.md`](../docs/SCOPE.md) — what the framework answers, what it doesn't (paper 4.1 falsifications listed explicitly)
- [`docs/EIGENSPACE_EVOLUTION.md`](../docs/EIGENSPACE_EVOLUTION.md) — relationship between v0.1 12-PC and v0.2 9-PC eigenspaces (read if comparing tool output to Paper 3)
- [`README.md`](../README.md) — full API reference and bundled data inventory
- GitHub issues — feature requests and bug reports
